# File Processing Simulation

This notebook demonstrates file input/output handling in sim2l.

In [ ]:
# Load sim2l IPython extension
%load_ext sim2l.notebook

import sim2l
print(f"sim2l version: {sim2l.__version__}")
print("✓ sim2l magics loaded")

## Define Inputs

File type inputs accept file paths.

In [ ]:
%%sim2l_inputs

input_file:
  type: Text
  description: "Path to input data file (CSV format)"

processing_mode:
  type: Text
  choices: ["summarize", "transform", "analyze"]
  default: "summarize"
  description: "Processing mode to apply"

output_format:
  type: Text
  choices: ["csv", "json", "txt"]
  default: "csv"
  description: "Output file format"

## Define Outputs

In [ ]:
%%sim2l_outputs

output_file:
  type: Text
  description: "Path to generated output file"

summary_report:
  type: Text
  description: "Path to summary report file"

row_count:
  type: Integer
  description: "Number of rows processed"

processing_time:
  type: Number
  units: second
  description: "Time taken to process"

## Setup

In [ ]:
import pandas as pd
import json as json_lib
import time
from pathlib import Path

print("✓ Imports loaded")

## Get Parameters

In [ ]:
# Get inputs
# When executed by Papermill, parameters are injected as variables
# When run interactively, use defaults

try:
    # Check if parameters were injected by Papermill
    _ = input_file
    print("Using Papermill-injected parameters")
except NameError:
    # Interactive mode - use defaults
    input_file = "sample_data.csv"
    processing_mode = "summarize"
    output_format = "csv"
    print(f"Using default test values")

print(f"\nParameters:")
print(f"  Input file: {input_file}")
print(f"  Processing mode: {processing_mode}")
print(f"  Output format: {output_format}")

## Process File

In [ ]:
start_time = time.time()

# Read input file
print(f"\nReading input file: {input_file}")
df = pd.read_csv(input_file)
row_count = len(df)
print(f"  Rows: {row_count}")
print(f"  Columns: {list(df.columns)}")

# Process based on mode
print(f"\nProcessing in '{processing_mode}' mode...")

if processing_mode == "summarize":
    # Generate summary statistics
    result_df = df.describe()
    summary_text = f"Summary Statistics:\n{result_df.to_string()}\n\n"
    summary_text += f"Data Types:\n{df.dtypes.to_string()}\n\n"
    summary_text += f"Missing Values:\n{df.isnull().sum().to_string()}"
    
elif processing_mode == "transform":
    # Apply transformations
    result_df = df.copy()
    # Normalize numeric columns
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        mean = df[col].mean()
        std = df[col].std()
        if std > 0:
            result_df[col] = (df[col] - mean) / std
    summary_text = f"Transformed {len(numeric_cols)} numeric columns (normalized)\n"
    summary_text += f"Columns: {', '.join(numeric_cols)}"
    
elif processing_mode == "analyze":
    # Perform analysis
    numeric_cols = df.select_dtypes(include=['number']).columns
    correlations = df[numeric_cols].corr() if len(numeric_cols) > 1 else None
    result_df = correlations if correlations is not None else df.describe()
    summary_text = f"Correlation Analysis:\n"
    if correlations is not None:
        summary_text += f"{correlations.to_string()}\n\n"
    else:
        summary_text += "Not enough numeric columns for correlation\n"
    summary_text += f"\nBasic Statistics:\n{df.describe().to_string()}"

print(f"  ✓ Processing complete")

## Generate Output Files

In [ ]:
# Generate output file
output_filename = f"output.{output_format}"

print(f"\nGenerating output file: {output_filename}")

if output_format == "csv":
    result_df.to_csv(output_filename, index=True)
elif output_format == "json":
    result_df.to_json(output_filename, orient='records', indent=2)
elif output_format == "txt":
    with open(output_filename, 'w') as f:
        f.write(result_df.to_string())

print(f"  ✓ Saved to {output_filename}")

# Generate summary report
report_filename = "summary_report.txt"
with open(report_filename, 'w') as f:
    f.write(f"File Processing Report\n")
    f.write(f"{'='*60}\n\n")
    f.write(f"Input File: {input_file}\n")
    f.write(f"Processing Mode: {processing_mode}\n")
    f.write(f"Output Format: {output_format}\n")
    f.write(f"Rows Processed: {row_count}\n\n")
    f.write(summary_text)

print(f"  ✓ Saved report to {report_filename}")

# Calculate processing time
processing_time = time.time() - start_time
print(f"\n✓ Processing completed in {processing_time:.2f}s")

## Save Outputs

In [ ]:
# Save outputs to database
sim2l.save_outputs(
    output_file=output_filename,
    summary_report=report_filename,
    row_count=int(row_count),
    processing_time=float(processing_time)
)

print("\n✓ Outputs saved to database!")

## Summary

In [ ]:
print("\n" + "="*60)
print("File Processing Complete!")
print("="*60)
print(f"Input:  {input_file} ({row_count} rows)")
print(f"Output: {output_filename}")
print(f"Report: {report_filename}")
print(f"Time:   {processing_time:.2f}s")
print("="*60)